# CSIRO Image2Biomass — CNN Training (Google Colab)

**Master's in Green Data Science — Practical ML**

This notebook trains the image models on a GPU. It mounts Google Drive, unpacks
the dataset to Colab's local disk (fast I/O), and runs the full cross-validated
comparison: from-scratch CNN, frozen ResNet (feature extraction), and fine-tuned
ResNet — against the tabular baseline.

**Before running:** set Runtime → Change runtime type → GPU (T4).

## Contents
1. GPU check
2. Mount Drive and unpack data
3. Install and import project
4. Smoke test
5. Full training (3 models × 5 folds)
6. Comparison with the tabular baseline
7. Save results

## 1. GPU check

Confirm a GPU is allocated. If this prints 'cpu', switch the runtime to GPU.

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi -L

CUDA available: False


'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


## 2. Mount Drive and unpack data

**One-time setup on your side:**
1. Zip the labelled images locally: the folder `data/raw/labelled/` -> `labelled.zip`.
2. Upload `labelled.zip` and `labelled.csv` to a Drive folder, e.g.
   `MyDrive/image2biomass/`.

The cell below mounts Drive and unpacks the zip into Colab's local disk
(`/content/...`) for fast image reading. Adjust `DRIVE_DIR` to your path.

In [4]:
from google.colab import drive
from pathlib import Path
import zipfile, shutil

drive.mount('/content/drive')

# >>> EDIT THIS to match where you put the files on Drive <<<
DRIVE_DIR = Path('/content/drive/MyDrive/image2biomass')

# Target layout on Colab local disk (matches config.in_colab() expectations)
PROJECT = Path('/content/image2biomass')
RAW = PROJECT / 'data' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)
(PROJECT / 'data' / 'processed').mkdir(parents=True, exist_ok=True)

# Copy CSV
shutil.copy(DRIVE_DIR / 'labelled.csv', RAW / 'labelled.csv')

# Unpack images to local disk (fast I/O vs reading from Drive directly)
with zipfile.ZipFile(DRIVE_DIR / 'labelled.zip') as z:
    z.extractall(RAW)

# Sanity: the images must end up at RAW/labelled/*.jpg
n_jpg = len(list((RAW / 'labelled').glob('*.jpg')))
print('labelled images on local disk:', n_jpg)
print('labelled.csv exists:', (RAW / 'labelled.csv').exists())

ModuleNotFoundError: No module named 'google.colab'

If `labelled images on local disk` is 0, the zip likely contains a nested
folder. Check with `!ls /content/image2biomass/data/raw` and move files so they
sit directly under `data/raw/labelled/`.

## 3. Install and import project

Upload your `src/` folder to the same Drive directory (or clone from GitHub).
The cell copies `src/` to the Colab project and imports the modules.
torch/torchvision are preinstalled on Colab.

In [ ]:
# Copy src/ from Drive to the Colab project
import shutil
shutil.copytree(DRIVE_DIR / 'src', PROJECT / 'src', dirs_exist_ok=True)

import sys
sys.path.insert(0, str(PROJECT / 'src'))

import importlib
import config, eda, splits, dataset, model, train
for m in (config, eda, splits, dataset, model, train):
    importlib.reload(m)

print("in_colab:", config.in_colab())
print("PROJECT_ROOT:", config.PROJECT_ROOT)
print("LABELLED_CSV exists:", config.LABELLED_CSV.exists())
print("device:", train.get_device())

## 4. Load data and smoke test

Quick 5-epoch run on frozen ResNet to confirm everything works on GPU before the full sweep.

In [ ]:
wide = eda.group_rare_species(eda.to_wide(eda.load_long()))
print("wide:", wide.shape)

smoke = train.cross_validate(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": True},
    n_splits=5, batch_size=32, num_workers=2,
    train_kwargs={"max_epochs": 5, "patience": 3},
    verbose=True,
)
print("smoke weighted RMSE:", round(smoke["weighted_rmse_mean"], 3))

## 5. Full training — 3 models × 5 folds

The three image models that map onto the course arc:
- **simple_cnn**: from scratch (session 10)
- **resnet18 frozen**: transfer learning, feature extraction (session 12)
- **resnet18 fine-tune**: transfer learning, full fine-tuning (session 12)

On a T4 this is a few minutes per model. Fine-tuning uses a smaller LR (1e-4)
since the whole backbone is updated.

In [ ]:
common = dict(n_splits=5, img_size=224, batch_size=32, num_workers=2, verbose=True)

results = {}

results["simple_cnn"] = train.cross_validate(
    wide, kind="simple_cnn", model_kwargs={},
    train_kwargs={"max_epochs": 60, "patience": 10, "lr": 1e-3}, **common)

results["resnet18_frozen"] = train.cross_validate(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": True},
    train_kwargs={"max_epochs": 40, "patience": 8, "lr": 1e-3}, **common)

results["resnet18_finetune"] = train.cross_validate(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": False},
    train_kwargs={"max_epochs": 30, "patience": 8, "lr": 1e-4}, **common)

for name, r in results.items():
    print(f"{name:22} weighted RMSE = {r['weighted_rmse_mean']:.3f} "
          f"± {r['weighted_rmse_std']:.3f}")

## 6. Comparison with the tabular baseline

The central table of the project: do image models beat the metadata-only baseline (RF weighted RMSE = 11.83, naive = 24.12)?

In [ ]:
import pandas as pd

rows = [
    {"model": "naive_median (tabular)", "weighted_rmse": 24.117},
    {"model": "RandomForest (tabular)", "weighted_rmse": 11.828},
]
for name, r in results.items():
    rows.append({"model": name + " (image)",
                 "weighted_rmse": round(r["weighted_rmse_mean"], 3)})
comparison = pd.DataFrame(rows).sort_values("weighted_rmse").reset_index(drop=True)
comparison

In [ ]:
# Per-target R2 of the best image model vs the metadata baseline
best_name = min(results, key=lambda k: results[k]["weighted_rmse_mean"])
print("Best image model:", best_name)
results[best_name]["summary"]

## 7. Save results

Persist metrics back to Drive so they survive a disconnect.

In [ ]:
import json
out = {name: {"weighted_rmse_mean": r["weighted_rmse_mean"],
              "weighted_rmse_std": r["weighted_rmse_std"]}
       for name, r in results.items()}
save_path = DRIVE_DIR / "cnn_results.json"
with open(save_path, "w") as f:
    json.dump(out, f, indent=2)
comparison.to_csv(DRIVE_DIR / "model_comparison.csv", index=False)
print("Saved to", save_path)

## Notes / conclusions

*(Fill in after the run.)*
- Does fine-tuning beat the tabular baseline (11.83)? If not, the conclusion is
  that cheap ground measurements (NDVI, height) outperform image models on 357
  samples — a meaningful, honest finding.
- frozen vs fine-tune gap: how much adapting the backbone helps.
- simple_cnn vs pretrained: the value of transfer learning on small data.
- Per-target: the image may help most where metadata is weak (Dry_Dead_g).